In [1]:
# CASSANDRA setup cell — run this FIRST, in order, before any other cell.
# On Colab with GPU desired (Pythia extraction): Runtime > Change runtime
# type > GPU, THEN run this cell.
import importlib
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "{repo_url}"

if IN_COLAB:
    if not Path("/content/CASSANDRA/.git").exists():
        subprocess.run(["git", "clone", REPO_URL, "/content/CASSANDRA"],
                       check=True)
    os.chdir("/content/CASSANDRA")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "spacy", "download",
                    "en_core_web_sm"], check=True, capture_output=True)
    subprocess.run([sys.executable, "scripts/prepare_data.py",
                    "--config", "config.yaml"], check=True)
    print("Colab setup complete. Working directory:", os.getcwd())
else:
    try:
        import torch, transformer_lens, sklearn, pandas, seaborn  # noqa: F401
        print("Local environment OK. Run `make data` if data/processed "
              "is empty.")
    except ImportError as exc:
        raise SystemExit(
            f"Missing dependency ({{exc}}). Run `make setup` from the "
            "repository root before executing this notebook."
        )


/home/z/my-project/cassandra/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Local environment OK. Run `make data` if data/processed is empty.


# 03 — Probe evaluation

**Objective.** The fine-grained statistical case for and against the probes: ROC and precision-recall with bootstrap uncertainty, calibration, the paired comparison against the model's own |logprob| baseline, and the Holm-Bonferroni correction over the pre-registered comparison family.

**Data.** Out-of-fold predictions from `logs/oof_predictions.jsonl` (every statement appears exactly once as a test example) and the aggregated table in `logs/probe_results.jsonl`.

**Honesty rules in effect.** AUC above 0.95 triggers a documented leakage audit; a comparison is only claimed when it survives Holm and its confidence interval excludes zero.

## Environment and determinism

In [2]:
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

pd.set_option("display.max_colwidth", 120)


In [3]:
import sys

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from cassandra_ns.report.figures import save_figure

if "google.colab" not in sys.modules:
    matplotlib.use("Agg")  # headless execution; Colab keeps its inline backend

sns.set_theme(context="notebook", style="whitegrid")
plt.rcParams["figure.dpi"] = 110


In [4]:
import subprocess
from pathlib import Path

def _ensure(artifact: str, command: list[str]) -> None:
    if not Path(artifact).exists():
        print(f"producing {artifact} via:", " ".join(command))
        subprocess.run([sys.executable, *command], check=True)

_ensure("data/processed/folds.json",
        ["scripts/prepare_data.py", "--config", "config.yaml"])


## Load out-of-fold predictions and results

In [5]:
from cassandra_ns.config import load_config
from cassandra_ns.data.loader import load_processed
from cassandra_ns.probes.evaluate import load_oof, load_results

CFG = load_config("config.yaml")
oof = load_oof(CFG)
results = load_results(CFG)
df = load_processed(CFG)  # domain of each statement for the per-domain curves
results.head()

,model,layer,mode,ptype,baseline_auc,n,auc,ci_lo,ci_hi,ap,ece,brier,p_baseline,diff,diff_lo,diff_hi,domain
0,gpt2,2,last_token,logistic,0.432832,6085,0.617393,0.603052,0.631149,0.596354,0.080329,0.254657,0.0,0.184561,0.162227,0.205302,None
1,gpt2,2,last_token,logistic,0.398924,1008,0.643707,0.610454,0.676295,0.673575,0.079990,0.226241,0.0,0.244784,0.194538,0.296203,animals
2,gpt2,2,last_token,logistic,0.360778,1458,0.469636,0.439530,0.499798,0.470228,0.195432,0.337326,0.0,0.108859,0.066190,0.147451,cities
3,gpt2,2,last_token,logistic,0.401736,1200,0.801178,0.779309,0.825315,0.805739,0.035830,0.181803,0.0,0.399442,0.355358,0.444658,companies
4,gpt2,2,last_token,logistic,0.478904,930,0.604722,0.567832,0.637406,0.594365,0.091865,0.251677,0.0,0.125818,0.066994,0.177654,elements


## Best configuration per model

Highest AUC configuration (layer, feature mode, probe type) per model after the grouped 5-fold CV. This is the configuration used for scoring generations in notebook 04.

In [6]:
best = (results[results["domain"].isna()]
          .sort_values("auc", ascending=False).groupby("model").head(1)
          .set_index("model"))
best[["layer", "mode", "ptype", "auc", "ci_lo", "ci_hi"]]

,layer,mode,ptype,auc,ci_lo,ci_hi
model,,,,,,
pythia-410m,11,last_token,mlp,0.795527,0.784188,0.807469
gpt2,5,mean_pool,mlp,0.728219,0.716068,0.740389


## ROC with bootstrap band

ROC of the best configuration per model; the band is the percentile bootstrap (1000 resamples, seed 42) of the curve on a fixed false-positive-rate grid, and the legend reports AUC with its 95% confidence interval.

In [7]:
from sklearn.metrics import roc_curve
from cassandra_ns.probes.stats import roc_auc, bootstrap_ci

FPR_GRID = np.linspace(0, 1, 101)

def _roc_band(y, s, n_boot=1000, seed=SEED):
    curves = []
    brng = np.random.default_rng(seed)
    for _ in range(n_boot):
        idx = brng.integers(0, len(y), size=len(y))
        fpr, tpr, _ = roc_curve(y[idx], s[idx])
        curves.append(np.interp(FPR_GRID, fpr, tpr))
    lo, hi = np.quantile(np.asarray(curves), [0.025, 0.975], axis=0)
    return lo, hi

fig, axes = plt.subplots(1, len(best), figsize=(5.6 * len(best), 4.6))
axes = np.atleast_1d(axes)
for ax, (model, row) in zip(axes, best.iterrows()):
    sel = oof[(oof["model"] == model) & (oof["layer"] == row["layer"])
              & (oof["mode"] == row["mode"]) & (oof["ptype"] == row["ptype"])]
    y = sel["y_true"].values.astype(int)
    s = sel["score"].values
    fpr, tpr, _ = roc_curve(y, s)
    lo, hi = _roc_band(y, s)
    ax.fill_between(FPR_GRID, lo, hi, alpha=0.18, color="#4c78a8",
                    label="95% bootstrap band")
    ax.plot(fpr, tpr, color="#4c78a8")
    ax.plot([0, 1], [0, 1], ls="--", c="grey", lw=1)
    _, ci_lo, ci_hi = bootstrap_ci(y, s, roc_auc, 1000, 0.95, SEED)
    ax.set_title(f"{model} — AUC {row['auc']:.3f} "
                 f"[{ci_lo:.3f}, {ci_hi:.3f}]")
    ax.set_xlabel("false positive rate")
    ax.set_ylabel("true positive rate")
    ax.legend(loc="lower right")
fig.tight_layout()
save_figure(fig, "03_roc_best_config")
plt.show()

## Precision-recall by domain

Out-of-fold precision-recall of the best configuration, one curve per domain. Domains with harder or noisier labels pull the pooled curve down; the spread of per-domain curves is the visual form of the out-of-distribution fragility discussed in the limits section.

In [8]:
from sklearn.metrics import precision_recall_curve, average_precision_score

row = best.iloc[0]
model = best.index[0]
sel = oof[(oof["model"] == model) & (oof["layer"] == row["layer"])
          & (oof["mode"] == row["mode"]) & (oof["ptype"] == row["ptype"])]
sel = sel.merge(df[["id", "domain"]], on="id")

fig, ax = plt.subplots(figsize=(6.8, 5.0))
for domain, group in sel.groupby("domain"):
    y, s = group["y_true"].values, group["score"].values
    prec, rec, _ = precision_recall_curve(y, s)
    ap = average_precision_score(y, s)
    ax.plot(rec, prec, label=f"{domain} (AP {ap:.2f})")
ax.set_xlabel("recall")
ax.set_ylabel("precision")
ax.set_title(f"{model}: precision-recall by domain")
ax.legend(fontsize=8)
fig.tight_layout()
save_figure(fig, "03_pr_by_domain")
plt.show()

## Calibration

Reliability diagrams for the best configuration per model with ECE (10 equal-width bins) and the Brier score, computed with the anchor statistics module. Probes are trained for ranking (AUC), not for probability reporting; the diagram shows how far their scores sit from usable probabilities.

In [9]:
from cassandra_ns.probes.calibration import reliability_diagram

fig, axes = plt.subplots(1, len(best), figsize=(5.2 * len(best), 4.8))
axes = np.atleast_1d(axes)
for ax, (model, row) in zip(axes, best.iterrows()):
    sel = oof[(oof["model"] == model) & (oof["layer"] == row["layer"])
              & (oof["mode"] == row["mode"]) & (oof["ptype"] == row["ptype"])]
    fig_d, info = reliability_diagram(sel["y_true"].values,
                                      sel["score"].values, ax=ax)
    ax.set_title(f"{model} — ECE {info['ece']:.3f}, "
                 f"Brier {info['brier']:.3f}")
fig.tight_layout()
save_figure(fig, "03_reliability_diagrams")
plt.show()

## Paired bootstrap: probe vs |logprob| baseline

The baseline is the model's own answer: the mean per-token logprob of the statement under the same model that produced the activations (score used: |logprob|). The bootstrap is paired — identical resampled indices for both systems — so the correlation between probe and baseline is preserved. The reported p-value and the decision come from the anchor statistics, not from this plot.

In [10]:
from cassandra_ns.probes.stats import (paired_bootstrap_diff_auc,
                                       roc_auc, bootstrap_ci)

fig, axes = plt.subplots(1, len(best), figsize=(5.6 * len(best), 4.4))
axes = np.atleast_1d(axes)
paired_rows = []
for ax, (model, row) in zip(axes, best.iterrows()):
    sel = oof[(oof["model"] == model) & (oof["layer"] == row["layer"])
              & (oof["mode"] == row["mode"]) & (oof["ptype"] == row["ptype"])]
    y = sel["y_true"].values.astype(int)
    probe_scores = sel["score"].values
    baseline = np.abs(sel["baseline"].values)
    res = paired_bootstrap_diff_auc(y, probe_scores, baseline,
                                    CFG["stats"]["n_bootstrap"],
                                    CFG["stats"]["ci"], SEED)
    paired_rows.append({"model": model, **res})
    brng = np.random.default_rng(SEED)
    diffs = []
    for _ in range(CFG["stats"]["n_bootstrap"]):
        idx = brng.integers(0, len(y), size=len(y))
        diffs.append(roc_auc(y[idx], probe_scores[idx])
                     - roc_auc(y[idx], baseline[idx]))
    ax.hist(diffs, bins=40, color="#4c78a8", alpha=0.85)
    ax.axvline(0, color="grey", ls="--", lw=1)
    ax.axvline(res["diff"], color="#b0553a", lw=2)
    ax.set_title(f"{model} — diff {res['diff']:.3f} "
                 f"[{res['lo']:.3f}, {res['hi']:.3f}], "
                 f"p={res['p_two_sided']:.4f}")
    ax.set_xlabel("AUC(probe) - AUC(|logprob|)")
fig.tight_layout()
save_figure(fig, "03_paired_bootstrap")
plt.show()

pd.DataFrame(paired_rows)

,model,diff,lo,hi,p_two_sided,n_resamples_skipped
0,pythia-410m,0.395698,0.378266,0.414391,0.0,0
1,gpt2,0.295387,0.276221,0.315171,0.0,0


## Multiple comparisons: Holm-Bonferroni

The comparison family was fixed in advance (layers x feature modes x probe types x models). Adjusted p-values for the probe-vs-baseline question across that family; only rows whose Holm-adjusted p falls below 0.05 AND whose confidence interval excludes zero may be claimed as improvements.

In [11]:
from cassandra_ns.probes.stats import holm_bonferroni

family = results[results["domain"].isna()].reset_index(drop=True)
pvals = family["p_baseline"].tolist()
family["p_holm"] = holm_bonferroni(pvals)
# the honesty rule binds the COMPARISON to a CI excluding zero: the paired
# difference CI (probe - baseline), not the AUC CI
family["ci_excludes_zero"] = (family["diff_lo"] > 0) | (family["diff_hi"] < 0)
family["claimable"] = (family["p_holm"] < 0.05) & family["ci_excludes_zero"]
family[["model", "layer", "mode", "ptype", "auc", "p_baseline",
        "p_holm", "claimable"]]

,model,layer,mode,ptype,auc,p_baseline,p_holm,claimable
0,gpt2,2,last_token,logistic,0.617393,0.0,0.0,True
1,gpt2,2,last_token,mlp,0.604896,0.0,0.0,True
2,gpt2,2,mean_pool,logistic,0.647840,0.0,0.0,True
3,gpt2,2,mean_pool,mlp,0.679333,0.0,0.0,True
4,gpt2,5,last_token,logistic,0.698092,0.0,0.0,True
5,gpt2,5,last_token,mlp,0.703844,0.0,0.0,True
6,gpt2,5,mean_pool,logistic,0.701567,0.0,0.0,True
7,gpt2,5,mean_pool,mlp,0.728219,0.0,0.0,True
8,gpt2,8,last_token,logistic,0.722042,0.0,0.0,True
9,gpt2,8,last_token,mlp,0.722040,0.0,0.0,True


## Summary table for the report

The README-facing table, built by `probe_summary_table` from the executed results only.

In [12]:
from cassandra_ns.report.tables import probe_summary_table

probe_summary_table(results)

,model,layer,mode,ptype,auc_ci,ap,ece,brier,baseline_auc,p_baseline,p_holm,claimable
0,gpt2,5,mean_pool,mlp,"0.728 [0.716, 0.740]",0.7369,0.0614,0.2136,0.4328,0.0,0.0,True
1,gpt2,8,mean_pool,mlp,"0.723 [0.711, 0.736]",0.7331,0.1009,0.2226,0.4328,0.0,0.0,True
2,gpt2,8,last_token,logistic,"0.722 [0.710, 0.735]",0.7301,0.0164,0.2112,0.4328,0.0,0.0,True
3,gpt2,8,last_token,mlp,"0.722 [0.710, 0.734]",0.7354,0.0465,0.2126,0.4328,0.0,0.0,True
4,gpt2,5,last_token,mlp,"0.704 [0.691, 0.717]",0.7182,0.0997,0.2288,0.4328,0.0,0.0,True
5,gpt2,11,last_token,mlp,"0.703 [0.691, 0.716]",0.7119,0.0716,0.2218,0.4328,0.0,0.0,True
6,gpt2,11,last_token,logistic,"0.703 [0.690, 0.717]",0.7074,0.0233,0.2178,0.4328,0.0,0.0,True
7,gpt2,5,mean_pool,logistic,"0.702 [0.690, 0.714]",0.7064,0.0236,0.2186,0.4328,0.0,0.0,True
8,gpt2,5,last_token,logistic,"0.698 [0.686, 0.712]",0.7039,0.0209,0.2197,0.4328,0.0,0.0,True
9,gpt2,8,mean_pool,logistic,"0.687 [0.676, 0.700]",0.6940,0.0285,0.2224,0.4328,0.0,0.0,True


## Interpretation: what survives the correction

- **Read the table above, not the point estimates.** After Holm correction and with 95% intervals, several layer x mode combinations may lose significance even when their raw AUC looks attractive; the pre-registered family exists precisely to prevent picking winners after the fact.
- **Against the baseline.** The probe must beat the model's own |logprob| to argue for reading internal states; a null paired result is a legitimate finding, not a failed experiment.
- **Leakage gate.** Any AUC above 0.95 in the results table would demand the documented entity-leakage audit before this notebook's conclusions are used anywhere.
- **Calibration caveat.** ECE and Brier quantify score quality as probabilities; abstention thresholds in notebook 04 are rank-based on purpose.
- **Scope.** All statements here are correlational and in-distribution: six domains, two small models, one dataset family.